In [80]:
# import libraries
import urllib.request
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import numpy as np
import os
path = r"C:\Users\Quyen\OneDrive - GNS Science\Offline_work\11_Github\housepriceprediction"
chrome_path = r"C:\Users\Quyen\OneDrive - GNS Science\Offline_work\11_Github\chromedriver-win64\chromedriver.exe"
os.chdir(path)
pd.set_option('display.max_columns', None)


In [13]:
def get_html_data(url, driver):
    driver.get(url)
    # execute script to scroll down the page
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);var lenOfPage=document.body.scrollHeight;return lenOfPage;")
    time.sleep(3)  
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup

In [16]:
# specify the url
urlpage = 'https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true'
print(urlpage)
# run firefox webdriver from executable path of your choice
service = Service(executable_path=chrome_path)
driver = webdriver.Chrome(service=service)
# get web page
driver.get(urlpage)
# execute script to scroll down the page
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);var lenOfPage=document.body.scrollHeight;return lenOfPage;")
# sleep for 30s
time.sleep(3)
driver.quit()

https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true


In [15]:
#Search for page containers
page_container = driver.find_element(By.CSS_SELECTOR, "div[data-test='paginated-items']")
page_numbers = []
if page_container:
    all_links = page_container.find_elements(By.TAG_NAME, "a")
    for link in all_links:
        page_num = link.text.strip()
        page_numbers.append(page_num)
# Filter only numeric page numbers and convert to int
page_nums_int = [int(p) for p in page_numbers if p.isdigit()]
min_page = min(page_nums_int) if page_nums_int else None
max_page = max(page_nums_int) if page_nums_int else None
print("Min page:", min_page)
print("Max page:", max_page)

# Base url 
base_url =  'https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true' 
# Creater master url 
master_url_dict = {}
master_rows = []
for page_num in range(min_page, max_page + 1):
    url = f"{base_url}&page={page_num}"
    print(f"Processing page: {url}")
    master_url_dict[page_num] = url


Min page: 1
Max page: 99
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=1
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=2
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=3
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=4
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=5
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=6
Processing page: https://www.realestate.co.nz/residential/sold/otago/dunedin-city?by=latest-sale&fby=sold-price-available&incs=true&page=7
Pr

In [55]:
service = Service(executable_path=chrome_path)
driver = webdriver.Chrome(service=service)
# create empty array to store data
data = []
for page_num in range(min_page, max_page + 1):
    page_url = master_url_dict[page_num]

    # get web page
    driver.get(page_url)
    time.sleep(5)  
    # Load the page
    driver.get(page_url)

    # Slow scroll to load all listings
    scroll_pause_time = 1.5
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(scroll_pause_time)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # Get HTML and parse with BeautifulSoup
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # Find all <a> with class containing 'listing-tile-info'
    results = soup.select('a[class*="listing-tile-info"]')
    print(f'Page {page_num} - Number of results: {len(results)}')

    for result in results:
        try:
            product_name = result.get_text(strip=True)
            product_link = result.get("href")
            if product_link and product_link.startswith("/"):
                product_link = "https://www.realestate.co.nz" + product_link
            data.append({
                "page": page_num,
                "product": product_name,
                "link": product_link
            })
        except Exception as e:
            print("Error parsing listing:", e)
    #driver.quit()
data_final = pd.DataFrame(data)
data_final.to_csv('HousepriceURLSolddunedin.csv') 

Page 1 - Number of results: 20
Page 2 - Number of results: 20
Page 3 - Number of results: 20
Page 4 - Number of results: 20
Page 5 - Number of results: 20
Page 6 - Number of results: 20
Page 7 - Number of results: 20
Page 8 - Number of results: 20
Page 9 - Number of results: 20
Page 10 - Number of results: 20
Page 11 - Number of results: 20
Page 12 - Number of results: 20
Page 13 - Number of results: 20
Page 14 - Number of results: 20
Page 15 - Number of results: 20
Page 16 - Number of results: 20
Page 17 - Number of results: 20
Page 18 - Number of results: 20
Page 19 - Number of results: 20
Page 20 - Number of results: 20
Page 21 - Number of results: 20
Page 22 - Number of results: 20
Page 23 - Number of results: 20
Page 24 - Number of results: 20
Page 25 - Number of results: 20
Page 26 - Number of results: 20
Page 27 - Number of results: 20
Page 28 - Number of results: 20
Page 29 - Number of results: 20
Page 30 - Number of results: 20
Page 31 - Number of results: 20
Page 32 - Number 

In [57]:
data_final = pd.read_csv('HousepriceURLSolddunedin.csv') 

In [75]:
def get_html_data(url, driver):
    driver.get(url)
    time.sleep(np.clip(np.random.lognormal(0, 0.5), 1, 5))  # between 1 and 5 seconds
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup

def get_text_by_data_test(soup, tag, data_test):
    try:
        el = soup.find(tag, attrs={"data-test": data_test})
        return el.get_text(strip=True) if el else None
    except:
        return None

def get_soldprice_and_date(soup):
    result = {}

    # --- 1. Get sold price ---
    container = soup.find("div", attrs={"data-test": "property-pricing-method"})
    if container:
        label = container.find("p", attrs={"data-test": "pricing-method__label"})
        if label and "Sold price" in label.text:
            price_tag = container.find("h3", attrs={"data-test": "pricing-method__price"})
            if price_tag:
                result["sold_price"] = price_tag.get_text(strip=True)

    # --- 2. Get last sold date ---
    sold_date_tag = soup.find("h3", attrs={"data-test": "property-pricing-info__label"})
    if sold_date_tag and "Last sold on" in sold_date_tag.text:
        # Extract just the date part
        date_text = sold_date_tag.get_text(strip=True).replace("Last sold on", "").strip()
        result["sold_date"] = date_text

    return result
def get_capital_value_and_date(soup):
    container = soup.find("div", attrs={"data-test-reinz-valuation-badge": True})
    if not container:
        return {}

    updated_date = container.find("p").get_text(strip=True).replace("Updated", "").strip()
    capital_value = container.find("h4", attrs={"data-test-reinz-valuation-badge-value": True}).get_text(strip=True)

    return {
        "capital_value": capital_value,
        "updated_date": updated_date
    }

def get_estimated_values(soup):
    results = []
    sections = soup.find_all("div", attrs={"data-test-reinz-valuation-badge": True})
    for section in sections:
        try:
            confidence = section.find("p").get_text(strip=True)
            value = section.find("h4", attrs={"data-test-reinz-valuation-badge-value": True}).get_text(strip=True)
            results.append({"confidence_band": confidence, "estimated_value": value})
        except:
            results.append({"confidence_band": None, "estimated_value": None})
    return results

def capital_value_detail(soup):
    container = soup.find("div", {"data-test": "capital-valuation"})
    if not container:
        return {}

    capital_value = container.find("h4").get_text(strip=True)
    date_text = container.find("div", class_="opacity-50").get_text(" ", strip=True)
    valuation_date = date_text.replace("Valuation date:", "").strip()

    values = container.find_all("div", class_="font-semibold")
    land_value = values[0].get_text(strip=True) if len(values) > 0 else None
    improvement_value = values[1].get_text(strip=True) if len(values) > 1 else None

    return {
        "capital_value": capital_value,
        "valuation_date": valuation_date,
        "land_value": land_value,
        "improvement_value": improvement_value
    }

def get_feature(soup, feature_name):
    try:
        tag = soup.find("title", string=feature_name)
        if tag:
            return tag.find_parent("div", class_="flex items-center").find("span").get_text(strip=True)
        return np.nan
    except:
        return np.nan

In [83]:
# run firefox webdriver from executable path of your choice
service = Service(executable_path=chrome_path)
driver = webdriver.Chrome(service=service)

all_listings = []

for index, row in data_final.iterrows():
    print(index, row['product'], row['link'])
    name = row['product']
    url = row['link']
    try:
        soup = get_html_data(url, driver)
        listing_info = {
            "product": name,
            "url": url,
            "title": get_text_by_data_test(soup, "h1", "listing-title"),
            "subtitle": get_text_by_data_test(soup, "h2", "listing-subtitle"),
            "description": get_text_by_data_test(soup, "div", "description-content__description"),
            "pricing_method": get_text_by_data_test(soup, "div", "listing-pricing-method"),
        }

        listing_info.update(get_capital_value_and_date(soup))
        listing_info.update(get_soldprice_and_date(soup))
        confidence_list = get_estimated_values(soup)
        for i, entry in enumerate(confidence_list, 1):
            listing_info[f"confidence_band_{i}"] = entry.get("confidence_band")
            listing_info[f"estimated_value_{i}"] = entry.get("estimated_value")

        listing_info.update(capital_value_detail(soup))

        features = ["Bedroom", "Bathroom", "Floor area", "Land area", "Title type", "Garage"]
        feature_data = {f.lower().replace(" ", "_"): get_feature(soup, f) for f in features}
        listing_info.update(feature_data)

        all_listings.append(listing_info)
    except Exception as e:
        print(f"Error loading {url}: {e}")
    continue
# Now all_listings is a list of dicts with combined data for all products
df_combined = pd.DataFrame(all_listings)
df_combined.to_csv("HousepriceSold.csv")

0 Last sold on
      28/05/20251 Pitt Street, North Dunedin1012m2$225,000(last sale price) https://www.realestate.co.nz/property/1-pitt-street-north-dunedin-dunedin-city-otago/sgrm35fy
1 Last sold on
      27/05/202558 Parklands Avenue, Mosgiel3$695,000(last sale price) https://www.realestate.co.nz/property/58-parklands-avenue-mosgiel-dunedin-city-otago/8vryf7ms
2 Last sold on
      23/05/2025267 Macandrew Road, Forbury31491m2$470,000(last sale price) https://www.realestate.co.nz/property/267-macandrew-road-forbury-dunedin-city-otago/y73lpc3r
3 Last sold on
      16/05/202521 Seaview Terrace, Kew31579m2$488,000(last sale price) https://www.realestate.co.nz/property/21-seaview-terrace-kew-dunedin-city-otago/2zwsplyr
4 Last sold on
      16/05/202530 Mcfadden Drive, Mosgiel31670m2$715,000(last sale price) https://www.realestate.co.nz/property/30-mcfadden-drive-mosgiel-dunedin-city-otago/vw93bzks
5 Last sold on
      16/05/202520 Holm Avenue, Broad Bay321362m2$1,200,000(last sale price) h

{'sold_price': '$225,000', 'sold_date': '28/05/2025'}